<a href="https://colab.research.google.com/github/changyunyeong/ComputerGraphics/blob/main/KNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [62]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import time, platform, pickle, os

from tensorflow.keras.datasets import cifar10
from tqdm.auto import tqdm

In [63]:
def load_pickle(f):
    version = platform.python_version_tuple()
    if version[0] == "2":
        return pickle.load(f)
    elif version[0] == "3":
        return pickle.load(f, encoding="latin1")
    raise ValueError("invalid python version: {}".format(version))

def load_CIFAR_batch(filename):
    """ load single batch of cifar """
    with open(filename, "rb") as f:
        datadict = load_pickle(f)
        X = datadict["data"]
        Y = datadict["labels"]
        X = X.reshape(10000, 3, 32, 32).transpose(0, 2, 3, 1).astype("float")
        Y = np.array(Y)
        return X, Y

def load_CIFAR10(ROOT):
    """ load all of cifar """
    xs = []
    ys = []
    for b in range(1, 6):
        f = os.path.join(ROOT, "data_batch_%d" % (b,))
        X, Y = load_CIFAR_batch(f)
        xs.append(X)
        ys.append(Y)
    Xtr = np.concatenate(xs)
    Ytr = np.concatenate(ys)
    del X, Y
    Xte, Yte = load_CIFAR_batch(os.path.join(ROOT, "test_batch"))
    return Xtr, Ytr, Xte, Yte

In [64]:
class KNN(object):
    def __init__(self, metric="L2"):
        self.metric = metric

    def train(self, X, y):
        self.Xtr = X.astype(np.float32)
        self.ytr = np.asarray(y).reshape(-1)

    def _distances(self, x):
        diff = self.Xtr - x.astype(np.float32)
        if self.metric == "L1":
            return np.sum(np.abs(diff), axis=1)
        return np.sqrt(np.sum(diff * diff, axis=1))

    def predict(self, X, k=1):

        X = X.astype(np.float32)
        num_test = X.shape[0]
        Ypred = np.zeros(num_test, dtype=self.ytr.dtype)

        for i in tqdm(range(num_test), desc=f"KNN k={k}, metric={self.metric}"):
            distances = self._distances(X[i])
            closest_indices = np.argpartition(distances, k - 1)[:k]
            closest_y = self.ytr[closest_indices].astype(int)
            Ypred[i] = np.argmax(np.bincount(closest_y))

        return Ypred

In [65]:
def run_knn(X_train, y_train, X_test, y_test, k=1, metric="L2"): # default distance metrics: L2
    knn = KNN(metric=metric)
    knn.train(X_train, y_train)
    y_pred = knn.predict(X_test, k=k)
    accuracy = np.mean(y_pred == y_test)
    return y_pred, accuracy

In [66]:
def confusion_counts_by_class(y_true, y_pred, labels=None):
    y_true = np.asarray(y_true).reshape(-1)
    y_pred = np.asarray(y_pred).reshape(-1)

    if labels is None:
        labels = np.unique(np.concatenate([y_true, y_pred]))

    rows = []
    for label in labels:
        true_is_label = y_true == label
        pred_is_label = y_pred == label
        tn = np.sum(~true_is_label & ~pred_is_label)
        tp = np.sum(true_is_label & pred_is_label)
        fn = np.sum(true_is_label & ~pred_is_label)
        fp = np.sum(~true_is_label & pred_is_label)
        rows.append([tn, tp, fn, fp])

    return pd.DataFrame(rows, index=labels, columns=["TN", "TP", "FN", "FP"])

In [67]:
Xtr, Ytr, Xte, Yte = load_CIFAR10('/')
num_train = 5000
num_test = 1000

# 5000 training images, 1000개 test images
Xtr_small = Xtr[:num_train]
Ytr_small = Ytr[:num_train]

Xte_small = Xte[:num_test]
Yte_small = Yte[:num_test]

# flattern images from 32x32x3 to 3072 features
Xtr_rows = Xtr_small.reshape(num_train, 32*32*3).astype(np.int16)
Xte_rows = Xte_small.reshape(num_test, 32*32*3).astype(np.int16)


In [68]:
distance_k_values = [1, 3]
metric_rows = []

for metric in ["L1", "L2"]:
    for k in distance_k_values:
        _, accuracy = run_knn(Xtr_rows, Ytr_small, Xte_rows, Yte_small, k=k, metric=metric)
        metric_rows.append({"metric": metric, "k": k, "accuracy": accuracy})

metric_results = pd.DataFrame(metric_rows)
display(metric_results)

best_metric_row = metric_results.sort_values("accuracy", ascending=False).iloc[0]
best_metric = best_metric_row["metric"]
print(f"Best distance metric: {best_metric} (k={int(best_metric_row['k'])}, accuracy={best_metric_row['accuracy']:.4f})")

KNN k=1, metric=L1:   0%|          | 0/1000 [00:00<?, ?it/s]

KNN k=3, metric=L1:   0%|          | 0/1000 [00:00<?, ?it/s]

KNN k=1, metric=L2:   0%|          | 0/1000 [00:00<?, ?it/s]

KNN k=3, metric=L2:   0%|          | 0/1000 [00:00<?, ?it/s]

,metric,k,accuracy
0,L1,1,0.288
1,L1,3,0.278
2,L2,1,0.268
3,L2,3,0.261


Best distance metric: L1 (k=1, accuracy=0.2880)


In [69]:
k_values = [1, 3, 5, 7, 9]
metrics = ["L1", "L2"]

all_results = []
all_predictions = {}

for metric in metrics:
    for k in k_values:
        y_pred, accuracy = run_knn(
            Xtr_rows,
            Ytr_small,
            Xte_rows,
            Yte_small,
            k=k,
            metric=metric
        )

        all_predictions[(metric, k)] = y_pred

        all_results.append({
            "metric": metric,
            "k": k,
            "accuracy": accuracy
        })

results_df = pd.DataFrame(all_results)
display(results_df)

best_row = results_df.sort_values("accuracy", ascending=False).iloc[0]
best_metric = best_row["metric"]
best_k = int(best_row["k"])
best_acc = best_row["accuracy"]

print(f"Best setting: metric={best_metric}, k={best_k}, accuracy={best_acc:.4f}")

best_prediction = all_predictions[(best_metric, best_k)]

KNN k=1, metric=L1:   0%|          | 0/1000 [00:00<?, ?it/s]

KNN k=3, metric=L1:   0%|          | 0/1000 [00:00<?, ?it/s]

KNN k=5, metric=L1:   0%|          | 0/1000 [00:00<?, ?it/s]

KNN k=7, metric=L1:   0%|          | 0/1000 [00:00<?, ?it/s]

KNN k=9, metric=L1:   0%|          | 0/1000 [00:00<?, ?it/s]

...

KNN k=5, metric=L2:   0%|          | 0/1000 [00:00<?, ?it/s]

KNN k=7, metric=L2:   0%|          | 0/1000 [00:00<?, ?it/s]

KNN k=9, metric=L2:   0%|          | 0/1000 [00:00<?, ?it/s]

,metric,k,accuracy
0,L1,1,0.288
1,L1,3,0.278
2,L1,5,0.310
3,L1,7,0.309
4,L1,9,0.301
5,L2,1,0.268
6,L2,3,0.261
7,L2,5,0.266
8,L2,7,0.274
9,L2,9,0.270


Best setting: metric=L1, k=5, accuracy=0.3100


In [72]:
class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer',
               'dog', 'frog', 'horse', 'ship', 'truck']
confusion_summary = confusion_counts_by_class(Yte_small, best_prediction, labels=np.arange(10))
confusion_summary.index = class_names
display(confusion_summary)


,TN,TP,FN,FP
airplane,775,56,47,122
automobile,898,16,73,13
bird,719,48,52,181
cat,858,21,82,39
deer,750,33,57,160
dog,889,12,74,25
frog,837,17,95,51
horse,882,17,85,16
ship,822,69,37,72
truck,880,21,88,11
